# Which Gases Contribute to Warming

### Imports and preprocessing

In [1]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

ROOT_DIR = Path.cwd().parent
Tables_DIR = ROOT_DIR / "data/tables"

# helper that saves each graph + summary so src/make_pdf.py can build the PDF
sys.path.insert(0, str(ROOT_DIR / "src"))
from report import save_report

TOPIC = "gases_warming"

gwpdf = pd.read_csv(Tables_DIR / "gwp_constants.csv")
country_gasesdf = pd.read_csv(Tables_DIR / "country_yearly.csv")

# Real countries only (drop OWID aggregates such as World, Asia, EU)
df = country_gasesdf.copy()
df = df[df["iso_code"].notna() & ~df["iso_code"].astype(str).str.startswith("OWID")]

BLUE, RED, GREEN, GREY = "#4C78A8", "#E45756", "#54A24B", "#9aa5b1"

def style(fig, title, sub, xt="", yt="", h=480, left=70, legend=True):
    fig.update_layout(
        template="plotly_white", width=1090,
        title=dict(text=f"<b>{title}</b><br><sup>{sub}</sup>",
                   x=0.02, xanchor="left", font=dict(color="#1f3b5c", size=20)),
        xaxis=dict(title=xt, gridcolor="#e8ecf3"),
        yaxis=dict(title=yt, gridcolor="#e8ecf3"),
        legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.2, title_text=""),
        showlegend=legend, height=h, margin=dict(l=left, r=40, t=90, b=90),
    )
    return fig

def rgba(hex_color, a=0.08):
    h = hex_color.lstrip("#")
    r, g_, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g_},{b},{a})"

print("countries rows:", df.shape, "| years", int(df["year"].min()), "-", int(df["year"].max()))
print(gwpdf)

countries rows: (42480, 25) | years 1750 - 2024
   gas  gwp20  gwp100  owid_gwp_used  \
0  CH4   82.5    29.8           29.8   
1  CO2    1.0     1.0            1.0   
2  N2O  273.0   273.0          273.0   

                                         source_note  
0  IPCC AR6 WGI Table 7.15 fossil methane (GWP20 ...  
1                             CO2 is 1 by definition  
2  IPCC AR6 WGI Table 7.15 (GWP20 273; GWP100 273...  


### Q1. Which gases contribute most to warming?

In [2]:
gases = {
    "co2_mt":        ("CO2",                 "#4C78A8"),
    "methane":       ("Methane (CH4)",       "#E45756"),
    "nitrous_oxide": ("Nitrous oxide (N2O)", "#54A24B"),
}

# Global total per gas per year (only years where all gases exist)
g = df.groupby("year")[list(gases)].sum(min_count=1).dropna()
y0, y1 = int(g.index.min()), int(g.index.max())

share_end = g.loc[y1] / g.loc[y1].sum() * 100
top_gas = gases[share_end.idxmax()][0]
subtitle = f"{top_gas} is {share_end.max():.0f}% of the three-gas total in {y1}"

# ---- graph ----
fig = go.Figure()
for key, (name, color) in gases.items():
    fig.add_trace(go.Scatter(
        x=g.index, y=g[key], name=name, mode="lines",
        line=dict(color=color, width=3), fill="tozeroy", fillcolor=rgba(color),
        hovertemplate="%{x}: %{y:,.0f} Mt<extra>" + name + "</extra>",
    ))
    pts = g.loc[[y0, y1], key]
    fig.add_trace(go.Scatter(
        x=pts.index, y=pts.values, mode="markers+text",
        marker=dict(color="#222", size=9), text=[f"{v:,.0f}" for v in pts.values],
        textposition="top center", showlegend=False, hoverinfo="skip",
    ))
style(fig, f"Which Gases Contribute Most to Warming? {y0}-{y1}", subtitle, xt="Year", yt="Emissions (Mt)")
fig.update_xaxes(range=[y0 - 3, y1 + 3])
fig.update_yaxes(rangemode="tozero")
fig.update_layout(hovermode="x unified")
fig.show()

print(f"Share of three-gas total in {y1}:")
print(share_end.rename({k: v[0] for k, v in gases.items()}).round(1).astype(str) + " %")

# ---- summary ----
rank = share_end.sort_values(ascending=False)
nm = [gases[k][0] for k in rank.index]
summary = (
    f"{nm[0]} is the largest contributor to warming. In {y1} it accounts for about {rank.iloc[0]:.1f}% of the "
    f"combined three-gas total (CO2, methane and nitrous oxide). {nm[1]} is second at {rank.iloc[1]:.1f}%, and "
    f"{nm[2]} is third at {rank.iloc[2]:.1f}%. Together, the two non-CO2 gases contribute about "
    f"{100 - share_end['co2_mt']:.0f}% of the total, so cutting CO2 matters most, but methane is the second lever. "
    f"These shares assume the methane and N2O columns are already in CO2-equivalent using the dataset's GWP100 factors."
)
print(summary)
save_report(TOPIC, "q1", "Which gases contribute most to warming?", fig, summary)

Share of three-gas total in 2024:
CO2                    75.0 %
Methane (CH4)          19.1 %
Nitrous oxide (N2O)     5.9 %
Name: 2024, dtype: str
CO2 is the largest contributor to warming. In 2024 it accounts for about 75.0% of the combined three-gas total (CO2, methane and nitrous oxide). Methane (CH4) is second at 19.1%, and Nitrous oxide (N2O) is third at 5.9%. Together, the two non-CO2 gases contribute about 25% of the total, so cutting CO2 matters most, but methane is the second lever. These shares assume the methane and N2O columns are already in CO2-equivalent using the dataset's GWP100 factors.
[report] saved gases_warming/q1 (1 graph(s))


### Q2. How does the picture change if you compare over 20 years instead of 100 years (GWP20 vs GWP100)?

In [3]:
cols = {"co2_mt": "CO2", "methane": "CH4", "nitrous_oxide": "N2O"}

# Latest year where all three gases are reported
g2 = df.groupby("year")[list(cols)].sum(min_count=1).dropna()
yr = int(g2.index.max())
co2e_100 = g2.loc[yr].rename(cols)          # already CO2e using the dataset's GWP factors

# Back out raw mass, then re-weight with GWP20 and GWP100
gw = gwpdf.set_index("gas")
raw = co2e_100 / gw["owid_gwp_used"]
gwp100 = raw * gw["gwp100"]
gwp20 = raw * gw["gwp20"]

res = pd.DataFrame({"GWP100 (100 years)": gwp100, "GWP20 (20 years)": gwp20})
share = res / res.sum() * 100

# ---- graph ----
colors = {"CO2": "#4C78A8", "CH4": "#E45756", "N2O": "#54A24B"}
fig = go.Figure()
for gas in share.index:
    fig.add_trace(go.Bar(
        y=share.columns, x=share.loc[gas], name=gas, orientation="h", marker_color=colors[gas],
        text=[f"{v:.1f}%" for v in share.loc[gas]], textposition="inside", insidetextanchor="middle",
        textfont=dict(color="white", size=14), customdata=res.loc[gas],
        hovertemplate="%{y}<br>" + gas + ": %{x:.1f}% (%{customdata:,.0f} Mt CO2e)<extra></extra>",
    ))
ch4_100, ch4_20 = share.loc["CH4"]
fig.update_layout(
    barmode="stack", template="plotly_white", width=1090, height=380,
    title=dict(text=f"<b>GWP20 vs GWP100: Share of Warming by Gas, {yr}</b>"
                    f"<br><sup>Methane rises from {ch4_100:.0f}% to {ch4_20:.0f}% when measured over 20 years</sup>",
               x=0.02, xanchor="left", font=dict(color="#1f3b5c", size=20)),
    xaxis=dict(title="Share of total warming (%)", range=[0, 100], gridcolor="#e8ecf3"),
    yaxis=dict(title="", autorange="reversed"),
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.25, title_text=""),
    margin=dict(l=150, r=40, t=90, b=90),
)
fig.show()

print(f"Year: {yr}")
print(share.round(1).astype(str) + " %")
print("\nTotal CO2e (Mt):")
print(res.sum().round(0))

# ---- summary ----
order100 = share["GWP100 (100 years)"].sort_values(ascending=False).index.tolist()
order20 = share["GWP20 (20 years)"].sort_values(ascending=False).index.tolist()
same_rank = ("The ranking stays the same (" + ", ".join(order100) + "), but the gap narrows sharply."
             if order100 == order20 else
             f"The ranking changes: {' > '.join(order100)} over 100 years, {' > '.join(order20)} over 20 years.")
summary = (
    f"{same_rank} Methane has a GWP of {gw.loc['CH4', 'gwp20']:.1f} over 20 years against "
    f"{gw.loc['CH4', 'gwp100']:.1f} over 100, because it traps heat strongly but breaks down in about a decade. "
    f"Using {yr} emissions as the baseline, methane's contribution rises from about {ch4_100:.0f}% to "
    f"{ch4_20:.0f}%, while CO2 falls from {share.loc['CO2', 'GWP100 (100 years)']:.0f}% to "
    f"{share.loc['CO2', 'GWP20 (20 years)']:.0f}%. N2O moves from {share.loc['N2O', 'GWP100 (100 years)']:.1f}% "
    f"to {share.loc['N2O', 'GWP20 (20 years)']:.1f}%, because the total grows when the short-lived gas is weighted more."
)
print(summary)
save_report(TOPIC, "q2",
            "How does the picture change if you compare over 20 years instead of 100 years (GWP20 vs GWP100)?",
            fig, summary)

Year: 2024
    GWP100 (100 years) GWP20 (20 years)
CH4             19.1 %           39.5 %
CO2             75.0 %           56.1 %
N2O              5.9 %            4.4 %

Total CO2e (Mt):
GWP100 (100 years)    49833.0
GWP20 (20 years)      66631.0
dtype: float64
The ranking stays the same (CO2, CH4, N2O), but the gap narrows sharply. Methane has a GWP of 82.5 over 20 years against 29.8 over 100, because it traps heat strongly but breaks down in about a decade. Using 2024 emissions as the baseline, methane's contribution rises from about 19% to 39%, while CO2 falls from 75% to 56%. N2O moves from 5.9% to 4.4%, because the total grows when the short-lived gas is weighted more.
[report] saved gases_warming/q2 (1 graph(s))


### Q3. Which emission sources dominate CO2 gas?

In [4]:
# Latest year with broad CO2 coverage (avoids a half-empty final year)
cover = df.groupby("year")["co2_mt"].count()
LATEST = int(cover[cover >= 100].index.max())
latest = df[df["year"] == LATEST].copy()

src = {"coal_co2": "Coal", "oil_co2": "Oil", "gas_co2": "Gas", "cement_co2": "Cement", "flaring_co2": "Flaring"}

tot = latest[list(src)].sum(min_count=1).rename(src)
src_share = (tot / tot.sum() * 100).sort_values()

# Compare with 50 years earlier
old_year = LATEST - 50
old = df[df["year"] == old_year][list(src)].sum(min_count=1).rename(src)
old_share = old / old.sum() * 100

# ---- graph ----
fig = go.Figure(go.Bar(
    y=src_share.index, x=src_share.values, orientation="h",
    marker_color=[RED if v == src_share.max() else BLUE for v in src_share.values],
    text=[f"{v:.1f}%" for v in src_share.values], textposition="outside",
    hovertemplate="%{y}: %{x:.1f}%<extra></extra>",
))
style(fig, f"Which Sources Dominate CO2 Emissions? {LATEST}",
      f"{src_share.idxmax()} is {src_share.max():.0f}% of fossil and industrial CO2",
      xt="Share of CO2 (%)", legend=False, h=400, left=100)
fig.update_xaxes(range=[0, src_share.max() * 1.2])
fig.show()

print(f"--- CO2 by source, {LATEST} ---")
out = pd.DataFrame({"Mt CO2": tot.round(0), f"share {LATEST} %": (tot / tot.sum() * 100).round(1),
                    f"share {old_year} %": old_share.round(1)}).sort_values("Mt CO2", ascending=False)
print(out)

# ---- summary ----
ranked = src_share.sort_values(ascending=False)
delta = (src_share - old_share).sort_values(ascending=False)
lead, second = ranked.index[0], ranked.index[1]
summary = (
    f"{lead} and {second} dominate, together producing about {ranked.iloc[:2].sum():.0f}% of fossil and "
    f"industrial CO2 in {LATEST}. {lead} is the largest single source at {ranked.iloc[0]:.1f}% "
    f"({tot[lead]:,.0f} Mt), compared with {old_share[lead]:.1f}% in {old_year}. {delta.index[0]} gained the most "
    f"share over the 50 years ({old_share[delta.index[0]]:.1f}% to {src_share[delta.index[0]]:.1f}%), while "
    f"{delta.index[-1]} lost the most ({old_share[delta.index[-1]]:.1f}% to {src_share[delta.index[-1]]:.1f}%). "
    f"{ranked.index[-2]} ({ranked.iloc[-2]:.1f}%) and {ranked.index[-1]} ({ranked.iloc[-1]:.1f}%) are minor sources. "
    f"The mix has shifted between fuels, which means power generation and industry as well as transport drive "
    f"the largest shares of CO2."
)
print(summary)
save_report(TOPIC, "q3", "Which emission sources dominate CO2 gas?", fig, summary)

--- CO2 by source, 2024 ---
          Mt CO2  share 2024 %  share 1974 %
Coal     15800.0          42.7          35.6
Oil      11276.0          30.5          46.3
Gas       8009.0          21.7          13.7
Cement    1473.0           4.0           2.1
Flaring    416.0           1.1           2.4
Coal and Oil dominate, together producing about 73% of fossil and industrial CO2 in 2024. Coal is the largest single source at 42.7% (15,800 Mt), compared with 35.6% in 1974. Gas gained the most share over the 50 years (13.7% to 21.7%), while Oil lost the most (46.3% to 30.5%). Cement (4.0%) and Flaring (1.1%) are minor sources. The mix has shifted between fuels, which means power generation and industry as well as transport drive the largest shares of CO2.
[report] saved gases_warming/q3 (1 graph(s))
